In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

folder = "/content/drive/MyDrive/Fraud_Project"

print(os.listdir(folder))

['firm_years.json', 'financial_with_aaer_labels.csv', 'aaer_label_quality_check.csv', 'matched_mda(1).csv', 'matched_mda.csv', 'clean_mda.csv']


In [ ]:
import pandas as pd

folder = "/content/drive/MyDrive/Fraud_Project"

financial_file = folder + "/financial_with_aaer_labels.csv"

df = pd.read_csv(financial_file)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

print("\nSample:")
display(df.head())

Rows: 40225
Columns:
['ticker', 'cik', 'fiscal_year', 'price', 'price_date', 'ddate', 'Assets', 'AssetsCurrent', 'CashAndCashEquivalentsAtCarryingValue', 'CommonStockSharesIssued', 'CommonStockValue', 'IncomeTaxExpenseBenefit', 'LiabilitiesCurrent', 'OperatingIncomeLoss', 'StockholdersEquity', 'NetIncomeLoss', 'Liabilities', 'is_fraud', 'fraud_label_source']

Sample:


,ticker,cik,fiscal_year,price,price_date,ddate,Assets,AssetsCurrent,CashAndCashEquivalentsAtCarryingValue,CommonStockSharesIssued,CommonStockValue,IncomeTaxExpenseBenefit,LiabilitiesCurrent,OperatingIncomeLoss,StockholdersEquity,NetIncomeLoss,Liabilities,is_fraud,fraud_label_source
0,A,1090872,2010,24.856937,2010-11-01,2010-10-31,9.696000e+09,6.169000e+09,2.649000e+09,579000000.0,6000000.0,8000000.0,3.083000e+09,5.660000e+08,3.228000e+09,NaN,6.460000e+09,0,No_AAER_fraud_period_overlap
1,A,1090872,2011,26.516453,2011-10-31,2011-10-31,9.057000e+09,5.569000e+09,3.527000e+09,591000000.0,6000000.0,20000000.0,1.837000e+09,1.071000e+09,4.308000e+09,NaN,4.741000e+09,0,No_AAER_fraud_period_overlap
2,A,1090872,2012,25.743919,2012-10-31,2012-10-31,1.053600e+10,4.629000e+09,2.351000e+09,595000000.0,6000000.0,-110000000.0,1.893000e+09,1.119000e+09,5.182000e+09,NaN,5.351000e+09,0,No_AAER_fraud_period_overlap
3,A,1090872,2013,36.309013,2013-10-31,2013-10-31,1.068600e+10,4.983000e+09,2.675000e+09,602000000.0,6000000.0,135000000.0,1.602000e+09,9.510000e+08,5.286000e+09,NaN,5.397000e+09,0,No_AAER_fraud_period_overlap
4,A,1090872,2014,39.542202,2014-10-31,2014-10-31,1.083100e+10,5.500000e+09,3.028000e+09,608000000.0,6000000.0,142000000.0,1.702000e+09,8.310000e+08,5.298000e+09,NaN,5.530000e+09,0,No_AAER_fraud_period_overlap


In [ ]:
!pip -q install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 2.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import ijson
import os
import re
from datetime import datetime

# ---------------------------------------------------------
# FILE LOCATIONS
# ---------------------------------------------------------

folder = "/content/drive/MyDrive/Fraud_Project"

financial_file = os.path.join(
    folder, "financial_with_aaer_labels.csv"
)

mda_file = os.path.join(
    folder, "firm_years.json"
)

output_file = os.path.join(
    folder, "matched_mda.csv"
)

# ---------------------------------------------------------
# 1. LOAD FINANCIAL DATA
# ---------------------------------------------------------

print("Loading financial dataset...")

financial = pd.read_csv(financial_file)

print("Financial rows:", len(financial))

# ---------------------------------------------------------
# 2. CREATE REQUIRED CIK + YEAR LOOKUP
# ---------------------------------------------------------

financial["cik_clean"] = (
    pd.to_numeric(financial["cik"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

financial["year_clean"] = (
    pd.to_numeric(financial["fiscal_year"], errors="coerce")
    .astype("Int64")
)

# Remove invalid values
financial_lookup = financial[
    financial["cik_clean"].notna() &
    financial["year_clean"].notna()
].copy()

# Create pairs we actually need
required_pairs = set(
    zip(
        financial_lookup["cik_clean"],
        financial_lookup["year_clean"].astype(int)
    )
)

print("Unique CIK-year pairs needed:", len(required_pairs))

# ---------------------------------------------------------
# 3. STREAM THROUGH THE 4.9 GB JSON
# ---------------------------------------------------------

print("\nStarting MD&A extraction...")
print("This may take some time because the JSON is 4.9 GB.")

matches = []

records_checked = 0

with open(mda_file, "rb") as f:

    # Assumes firm_years.json is a JSON array of records
    records = ijson.items(f, "item")

    for record in records:

        records_checked += 1

        # -----------------------------
        # CIK
        # -----------------------------
        cik = record.get("cik")

        if cik is None:
            continue

        try:
            cik_clean = str(int(float(cik)))
        except:
            continue

        # -----------------------------
        # REPORTING DATE
        # -----------------------------
        reporting_date = record.get("reporting_date")

        if not reporting_date:
            continue

        # Extract year
        match = re.search(r"(19|20)\d{2}", str(reporting_date))

        if not match:
            continue

        year = int(match.group())

        # -----------------------------
        # CHECK WHETHER WE NEED THIS
        # -----------------------------
        if (cik_clean, year) not in required_pairs:
            continue

        # -----------------------------
        # MD&A
        # -----------------------------
        mda = record.get("mda")

        if not mda:
            continue

        # Save useful information
        matches.append({
            "cik": cik_clean,
            "fiscal_year": year,
            "reporting_date": reporting_date,
            "mda": mda,
            "url": record.get("url")
        })

        # Progress every 100,000 records
        if records_checked % 100000 == 0:
            print(
                f"Checked: {records_checked:,} | "
                f"MD&A matches: {len(matches):,}"
            )

print("\nExtraction finished.")

print("Total JSON records checked:", records_checked)
print("MD&A records matched:", len(matches))

# ---------------------------------------------------------
# 4. SAVE MATCHED MD&A
# ---------------------------------------------------------

mda_df = pd.DataFrame(matches)

mda_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)

print("\nOutput shape:", mda_df.shape)

display(mda_df.head())

Loading financial dataset...
Financial rows: 40225
Unique CIK-year pairs needed: 40225

Starting MD&A extraction...
This may take some time because the JSON is 4.9 GB.

Extraction finished.
Total JSON records checked: 84203
MD&A records matched: 32587

Saved:
/content/drive/MyDrive/Fraud_Project/matched_mda.csv

Output shape: (32587, 5)


,cik,fiscal_year,reporting_date,mda,url
0,3453,2013,31-12-2013,ITEM 7. MANAGEMENT&#146;S DISCUSSION AND ANAL...,https://www.sec.gov/Archives/edgar/data/3453/0...
1,3499,2011,31-12-2011,ITEM 7. management&#8217;s discussion and ana...,https://www.sec.gov/Archives/edgar/data/3499/0...
2,3499,2012,31-12-2012,ITEM 7. management&#8217;s discussion and ana...,https://www.sec.gov/Archives/edgar/data/3499/0...
3,3499,2013,31-12-2013,ITEM 7. management&#8217;s discussion and ana...,https://www.sec.gov/Archives/edgar/data/3499/0...
4,3499,2015,31-12-2015,ITEM 7. management&#8217;s discussion and ana...,https://www.sec.gov/Archives/edgar/data/3499/0...


In [ ]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/Fraud_Project"
mda_file = os.path.join(folder, "matched_mda.csv")

mda = pd.read_csv(mda_file)

print("Rows:", len(mda))
print("Columns:", mda.columns.tolist())

print("\nDuplicate CIK-year pairs:")
print(
    mda.duplicated(
        subset=["cik", "fiscal_year"],
        keep=False
    ).sum()
)

print("\nMissing MD&A:")
print(mda["mda"].isna().sum())

print("\nEmpty MD&A:")
print(
    (mda["mda"].fillna("").str.strip() == "").sum()
)

print("\nMD&A character statistics:")
print(mda["mda"].fillna("").str.len().describe())

display(mda.head(3))

Rows: 32587
Columns: ['cik', 'fiscal_year', 'reporting_date', 'mda', 'url']

Duplicate CIK-year pairs:
1763

Missing MD&A:
0

Empty MD&A:
0

MD&A character statistics:
count    3.258700e+04
mean     7.894830e+04
std      5.853074e+04
min      7.000000e+00
25%      4.402950e+04
50%      6.841200e+04
75%      1.001480e+05
max      1.251239e+06
Name: mda, dtype: float64


,cik,fiscal_year,reporting_date,mda,url
0,3453,2013,31-12-2013,ITEM 7. MANAGEMENT&#146;S DISCUSSION AND ANAL...,https://www.sec.gov/Archives/edgar/data/3453/0...
1,3499,2011,31-12-2011,ITEM 7. management&#8217;s discussion and ana...,https://www.sec.gov/Archives/edgar/data/3499/0...
2,3499,2012,31-12-2012,ITEM 7. management&#8217;s discussion and ana...,https://www.sec.gov/Archives/edgar/data/3499/0...


In [ ]:
# Find duplicate CIK-year groups
dups = mda[
    mda.duplicated(
        subset=["cik", "fiscal_year"],
        keep=False
    )
].copy()

dups = dups.sort_values(
    ["cik", "fiscal_year", "reporting_date"]
)

print("Rows involved in duplicate groups:", len(dups))
print(
    "Number of duplicate CIK-year groups:",
    dups.groupby(["cik", "fiscal_year"]).ngroups
)

# Show examples
display(
    dups[
        ["cik", "fiscal_year", "reporting_date", "url"]
    ].head(30)
)

Rows involved in duplicate groups: 1763
Number of duplicate CIK-year groups: 857


,cik,fiscal_year,reporting_date,url
113,1800,2012,31-12-2012,https://www.sec.gov/Archives/edgar/data/1800/0...
116,1800,2012,31-12-2012,https://www.sec.gov/Archives/edgar/data/1800/0...
96,1961,2012,31-12-2012,https://www.sec.gov/Archives/edgar/data/1961/0...
97,1961,2012,31-12-2012,https://www.sec.gov/Archives/edgar/data/1961/0...
95,1961,2013,31-12-2013,https://www.sec.gov/Archives/edgar/data/1961/0...
100,1961,2013,31-12-2013,https://www.sec.gov/Archives/edgar/data/1961/0...
101,1961,2016,31-12-2016,https://www.sec.gov/Archives/edgar/data/1961/0...
102,1961,2016,31-12-2016,https://www.sec.gov/Archives/edgar/data/1961/0...
78,2186,2021,31-12-2021,https://www.sec.gov/Archives/edgar/data/2186/0...
80,2186,2021,31-12-2021,https://www.sec.gov/Archives/edgar/data/2186/0...


In [ ]:
# Inspect the shortest MD&A records
mda["mda_length"] = mda["mda"].fillna("").str.len()

shortest = mda.sort_values("mda_length").head(30)

display(
    shortest[
        ["cik", "fiscal_year", "reporting_date", "mda_length", "mda"]
    ]
)

,cik,fiscal_year,reporting_date,mda_length,mda
20744,1163739,2017,31-12-2017,7,The fo
25240,1410708,2012,31-07-2012,100,Item 7. Management's Discussion and Analysis ...
25239,1410708,2013,31-07-2013,100,Item 7. Management's Discussion and Analysis ...
25238,1410708,2013,31-07-2013,100,Item 7. Management's Discussion and Analysis ...
21961,1281761,2013,31-12-2013,103,Item 7. Management&#8217;s Discussion and Ana...
21960,1281761,2016,31-12-2016,103,Item 7. Management&#8217;s Discussion and Ana...
21962,1281761,2017,31-12-2017,103,Item 7. Management&#8217;s Discussion and Ana...
21967,1281761,2020,31-12-2020,103,Item 7. Management&#8217;s Discussion and Ana...
21957,1281761,2015,31-12-2015,103,Item 7. Management&#8217;s Discussion and Ana...
21959,1281761,2016,31-12-2016,103,Item 7. Management&#8217;s Discussion and Ana...


In [ ]:
# Inspect MD&A lengths below 1000 characters

short_mda = (
    mda[mda["mda_length"] < 1000]
    .sort_values("mda_length")
)

print("Records with MD&A < 1000 characters:", len(short_mda))

display(
    short_mda[
        ["cik", "fiscal_year", "reporting_date",
         "mda_length", "mda"]
    ].head(50)
)

Records with MD&A < 1000 characters: 1064


,cik,fiscal_year,reporting_date,mda_length,mda
20744,1163739,2017,31-12-2017,7,The fo
25238,1410708,2013,31-07-2013,100,Item 7. Management's Discussion and Analysis ...
25240,1410708,2012,31-07-2012,100,Item 7. Management's Discussion and Analysis ...
25239,1410708,2013,31-07-2013,100,Item 7. Management's Discussion and Analysis ...
21961,1281761,2013,31-12-2013,103,Item 7. Management&#8217;s Discussion and Ana...
21967,1281761,2020,31-12-2020,103,Item 7. Management&#8217;s Discussion and Ana...
21959,1281761,2016,31-12-2016,103,Item 7. Management&#8217;s Discussion and Ana...
21958,1281761,2014,31-12-2014,103,Item 7. Management&#8217;s Discussion and Ana...
21963,1281761,2018,31-12-2018,103,Item 7. Management&#8217;s Discussion and Ana...
21966,1281761,2019,31-12-2019,103,Item 7. Management&#8217;s Discussion and Ana...


In [ ]:
# Length distribution in more detail

print(
    mda["mda_length"].quantile(
        [0, .001, .005, .01, .02, .05, .10, .25, .50]
    )
)

0.000        7.0
0.001      111.0
0.005      258.0
0.010      300.0
0.020      382.0
0.050    10246.0
0.100    22003.8
0.250    44029.5
0.500    68412.0
Name: mda_length, dtype: float64


In [ ]:
import pandas as pd
import os
import re

folder = "/content/drive/MyDrive/Fraud_Project"

mda_file = os.path.join(folder, "matched_mda.csv")
output_file = os.path.join(folder, "clean_mda.csv")

# Load
mda = pd.read_csv(mda_file)

# Make sure text is string
mda["mda"] = mda["mda"].fillna("").astype(str)

# Calculate length
mda["mda_length"] = mda["mda"].str.len()

print("Original rows:", len(mda))

# ---------------------------------------------------------
# 1. Remove extremely short records
# ---------------------------------------------------------

# Anything below 300 characters is very likely to be
# an incomplete extraction / heading rather than full MD&A.
step1 = mda[mda["mda_length"] >= 300].copy()

print("After minimum-length filter:", len(step1))
print("Removed:", len(mda) - len(step1))

# ---------------------------------------------------------
# 2. Remove heading-only MD&A records
# ---------------------------------------------------------

def looks_like_heading(text):
    text = str(text).strip().lower()

    # Remove HTML entities / whitespace for checking
    normalized = re.sub(r"\s+", " ", text)

    # Common heading-only patterns
    patterns = [
        r"^item\s*[67]\.?\s*management['’`]?s discussion and analysis",
        r"^item\s*[67]\s+management['’`]?s discussion and analysis",
        r"^management['’`]?s discussion and analysis",
    ]

    for pattern in patterns:
        if re.search(pattern, normalized):
            # If the entire text is still very short,
            # treat it as heading-only
            if len(normalized) < 1000:
                return True

    return False


step2 = step1[
    ~step1["mda"].apply(looks_like_heading)
].copy()

print("After heading-only filter:", len(step2))
print("Removed:", len(step1) - len(step2))

# ---------------------------------------------------------
# 3. Remove duplicate CIK-year records
#    Keep longest valid MD&A
# ---------------------------------------------------------

step2 = step2.sort_values(
    ["cik", "fiscal_year", "mda_length"],
    ascending=[True, True, False]
)

clean_mda = (
    step2
    .drop_duplicates(
        subset=["cik", "fiscal_year"],
        keep="first"
    )
    .copy()
)

print("Final clean rows:", len(clean_mda))
print(
    "Duplicate rows removed:",
    len(step2) - len(clean_mda)
)

# ---------------------------------------------------------
# 4. Save
# ---------------------------------------------------------

clean_mda.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)

# ---------------------------------------------------------
# 5. Final statistics
# ---------------------------------------------------------

print("\nFinal MD&A length statistics:")
print(clean_mda["mda_length"].describe())

print("\nFinal duplicate CIK-year pairs:")
print(
    clean_mda.duplicated(
        subset=["cik", "fiscal_year"]
    ).sum()
)

display(
    clean_mda[
        ["cik", "fiscal_year",
         "reporting_date", "mda_length"]
    ].head(10)
)

Original rows: 32587
After minimum-length filter: 32265
Removed: 322
After heading-only filter: 32138
Removed: 127
Final clean rows: 31241
Duplicate rows removed: 897

Saved:
/content/drive/MyDrive/Fraud_Project/clean_mda.csv

Final MD&A length statistics:
count    3.124100e+04
mean     8.062509e+04
std      5.842775e+04
min      3.000000e+02
25%      4.565700e+04
50%      6.963800e+04
75%      1.013180e+05
max      1.251239e+06
Name: mda_length, dtype: float64

Final duplicate CIK-year pairs:
0


,cik,fiscal_year,reporting_date,mda_length
126,1750,2011,31-05-2011,33655
128,1750,2012,31-05-2012,44910
127,1750,2013,31-05-2013,37592
129,1750,2014,31-05-2014,43772
130,1750,2015,31-05-2015,52711
132,1750,2016,31-05-2016,50260
131,1750,2017,31-05-2017,47532
135,1750,2018,31-05-2018,51530
133,1750,2019,31-05-2019,59324
134,1750,2020,31-05-2020,54184


In [ ]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/Fraud_Project"

financial_file = os.path.join(
    folder, "financial_with_aaer_labels.csv"
)

mda_file = os.path.join(
    folder, "clean_mda.csv"
)

financial = pd.read_csv(financial_file)
mda = pd.read_csv(mda_file)

# Standardize CIK
financial["cik"] = (
    pd.to_numeric(financial["cik"], errors="coerce")
    .astype("Int64")
)

mda["cik"] = (
    pd.to_numeric(mda["cik"], errors="coerce")
    .astype("Int64")
)

# Standardize year
financial["fiscal_year"] = (
    pd.to_numeric(
        financial["fiscal_year"],
        errors="coerce"
    ).astype("Int64")
)

mda["fiscal_year"] = (
    pd.to_numeric(
        mda["fiscal_year"],
        errors="coerce"
    ).astype("Int64")
)

# Create CIK-year keys
financial["cik_year"] = (
    financial["cik"].astype(str)
    + "_"
    + financial["fiscal_year"].astype(str)
)

mda["cik_year"] = (
    mda["cik"].astype(str)
    + "_"
    + mda["fiscal_year"].astype(str)
)

# ---------------------------------------------------------
# MATCH
# ---------------------------------------------------------

financial_keys = set(financial["cik_year"])
mda_keys = set(mda["cik_year"])

matched_keys = financial_keys.intersection(mda_keys)

print("Financial company-years:", len(financial_keys))
print("Clean MD&A company-years:", len(mda_keys))
print("Matched company-years:", len(matched_keys))

print(
    "Financial company-years with MD&A:",
    len(matched_keys) / len(financial_keys) * 100,
    "%"
)

# ---------------------------------------------------------
# CHECK FRAUD LABELS IN MATCHED DATA
# ---------------------------------------------------------

matched_financial = financial[
    financial["cik_year"].isin(matched_keys)
].copy()

print("\nMatched financial rows:", len(matched_financial))

print("\nFraud label distribution:")
print(
    matched_financial["is_fraud"]
    .value_counts(dropna=False)
)

print("\nFraud percentage:")
print(
    matched_financial["is_fraud"].mean() * 100
)
